# 04 — Machine Learning

Notebook dedicato al preprocessing finale, alla selezione delle feature e all'addestramento dei modelli.

**Sprint 3 | Settimana 3**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

sys.path.append(os.path.abspath(".."))

df = pd.read_csv(os.path.abspath(os.path.join("..", "data", "processed", "data_clean.csv")))

sns.set_theme(style="whitegrid")
print(f"Dataset caricato: {df.shape[0]} righe, {df.shape[1]} colonne")

Dataset caricato: 7043 righe, 37 colonne


## Rimozione variabili ridondanti

Dall'analisi della matrice di correlazione (US-10) emergono 3 coppie dove una variabile
è sostanzialmente un duplicato dell'altra. Le rimuoviamo prima di qualsiasi modellazione.

| Rimossa | Mantenuta | Correlazione | Motivazione |
|---|---|---|---|
| `partner` | `married` | 1.000 | Identiche — stessa informazione codificata due volte |
| `dependents` | `num_dependents` | 0.888 | Binaria vs conteggio — `num_dependents` è più informativa |
| `total_charges` | `total_revenue` | 0.972 | `total_revenue` include già i rimborsi — più completa |

In [2]:
from src.preprocessing import drop_redundant_features

df = drop_redundant_features(df)
print(f"Colonne dopo rimozione ridondanti: {df.shape[1]}")
print(f"Colonne rimosse: partner, dependents, total_charges")

Colonne dopo rimozione ridondanti: 34
Colonne rimosse: partner, dependents, total_charges


## US-15 · Preprocessing delle Variabili

Le variabili vengono classificate in tre tipi e trattate con trasformazioni diverse:
- **Numeriche continue** → standardizzazione (`StandardScaler`)
- **Categoriche nominali** → One-Hot Encoding (`OneHotEncoder`)
- **Binarie (0/1)** → nessuna trasformazione necessaria

### Classificazione delle variabili

In [3]:
# Separazione target
y = df["churn"]
X = df.drop(columns=["churn"])

# Variabili categoriche nominali — richiedono One-Hot Encoding
cat_cols = ["contract", "internet_type", "payment_method", "offer", "gender"]

# Variabili binarie (già 0/1) — nessuna trasformazione
# partner e dependents già rimossi nella sezione precedente (ridondanti con married e num_dependents)
binary_cols = [
    "device_protection_plan", "internet_service", "married",
    "multiple_lines", "online_backup", "online_security", "paperless_billing",
    "phone_service", "premium_tech_support", "referred_a_friend",
    "senior_citizen", "streaming_movies", "streaming_music", "streaming_tv",
    "unlimited_data"
]

# Variabili numeriche continue — richiedono StandardScaler
# total_charges già rimossa nella sezione precedente (ridondante con total_revenue)
num_cols = [col for col in X.columns if col not in cat_cols + binary_cols]

print(f"Numeriche continue ({len(num_cols)}): {num_cols}")
print(f"\nCategoriche nominali ({len(cat_cols)}): {cat_cols}")
print(f"\nBinarie ({len(binary_cols)}): {binary_cols}")

Numeriche continue (13): ['age', 'avg_monthly_gb_download', 'avg_monthly_ld_charges', 'cltv', 'monthly_charge', 'num_dependents', 'num_referrals', 'population', 'tenure_months', 'total_extra_data_charges', 'total_ld_charges', 'total_refunds', 'total_revenue']

Categoriche nominali (5): ['contract', 'internet_type', 'payment_method', 'offer', 'gender']

Binarie (15): ['device_protection_plan', 'internet_service', 'married', 'multiple_lines', 'online_backup', 'online_security', 'paperless_billing', 'phone_service', 'premium_tech_support', 'referred_a_friend', 'senior_citizen', 'streaming_movies', 'streaming_music', 'streaming_tv', 'unlimited_data']


### Pipeline di preprocessing con ColumnTransformer

`ColumnTransformer` applica trasformazioni diverse a colonne diverse in un unico oggetto.
`Pipeline` concatena preprocessing e modello — garantisce che la standardizzazione
venga calcolata **solo sul train set** e poi applicata al test, evitando data leakage.

In [4]:
from src.preprocessing import build_preprocessor

preprocessor = build_preprocessor(num_cols, cat_cols, binary_cols)
print("✅ Preprocessor costruito")
print(preprocessor)

✅ Preprocessor costruito
ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['age', 'avg_monthly_gb_download',
                                  'avg_monthly_ld_charges', 'cltv',
                                  'monthly_charge', 'num_dependents',
                                  'num_referrals', 'population',
                                  'tenure_months', 'total_extra_data_charges',
                                  'total_ld_charges', 'total_refunds',
                                  'total_revenue']),
                                ('cat',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['co...t', 'internet_type', 'payment_method',
                                  'offer', 'gender']),
                                ('bin', 'passthrough',
                                 ['device_protection_plan', 'internet_ser

### Osservazioni US-15

| Tipo | N. variabili | Trasformazione | Motivazione |
|---|---|---|---|
| Numeriche continue | 13 | `StandardScaler` | Portare tutte le variabili sulla stessa scala — obbligatorio per LR e MLP |
| Categoriche nominali | 5 | `OneHotEncoder` | Convertire le categorie in colonne 0/1 — i modelli non interpretano stringhe |
| Binarie | 15 | Nessuna | Già in formato 0/1 |

**Nota sul data leakage nella pipeline**: usando `Pipeline(preprocessor, modello)` e addestrando
solo sul train set, lo scaler impara media e deviazione standard esclusivamente dai dati di train.
Quando trasformiamo il test set usiamo i parametri del train — mai i dati di test.